# Pipelines

Un Piepeline, usualmente se utiliza como una funcion que resume otras funciones. Crear un Pipeline puede ser dificil pero sus uso es bastante util, mantiene el codigo limpio y lo hace mas legible.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
data = pd.read_csv("data/melb_data.csv")
y = data.Price
X = data.drop(['Price'], axis = 1)
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state = 0)

# usaremos las funciones categoricas con baja cardinalidad
categorical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].nunique() < 10 and X_train_full[cname].dtype == "object"]
# conservamos las funciones numericas
numerical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].dtype in ['int64', 'float64']]

# ahora tomamos las columnas seleccionadas
my_cols = categorical_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()

## Definimos el proceso
El dataset que tenemos es uno al que le faltan datos y al que hay que codificar los valores categoricos.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# hacemos el preproceso de las columna numericas
numerical_transformer = SimpleImputer(strategy = "constant")
# preproceso para datos categoricos
categorical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = "most_frequent")),
    ('onehot', OneHotEncoder(handle_unknown = 'ignore'))
])

# otro metodo de preproceso para las columnas numericas y categoricas
preprocessor = ColumnTransformer(
    transformers = [
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

## Definimos el proceso

In [4]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators = 100, random_state = 0)

## Ajustamos y evaluamos 

In [5]:
from sklearn.metrics import mean_absolute_error

my_pipeline = Pipeline(steps = [('preprocessore', preprocessor),
                                ('model', model)])
my_pipeline.fit(X_train, y_train)
preds = my_pipeline.predict(X_valid)

score = mean_absolute_error(y_valid, preds)
print("mae: ", score)

mae:  160679.18917034855


Vemos que al procesar los datos categoricos y los datos nuemericos, nos da un MAE bajo a comparacion de solo hacer la prediccion si un preproceso previo.

# Cross Validation

En cross-validation, ejecutamos proceso de modelacion en diferentes subconjuntos de los datos para obtener multplies mediciones de la calidad del modelo. Es decir, supongamos que la informacion se divide en subconjuntos con el 25% de la informacion cada uno: A, B, C y D. Entonces, la informacion se entrenara con la informacion de la A hasta la C y el conjunto D dera de validacion. Luego, el modelo se le entrenara con la informacion de la B hasta la D y el conjunto A ser la de validacion, para depues hacer lo mismo en el que B y C quedan excluidas del conjunto de entrenamiento para ser los conjuntos de validacion.

![Cross-Validation](images/cross-val.png)

En pocas palabras, entrenaremos un modelo con la misma informacion paro ordenada de distinta forma por cada experimento.

In [12]:
from sklearn.model_selection import cross_val_score

# multiplicamos por -1 ya que sklearn calcula numeros negativos en MAE
scores = -1 * cross_val_score(my_pipeline, X, y, cv = 5, scoring = 'neg_mean_absolute_error')
print("MAE scores: ", scores)

MAE scores:  [207273.036228   195544.72890525 186907.59467108 152084.99219493
 158236.49133232]


In [13]:
print("Average MAE score (across experiments):")
print(scores.mean())

Average MAE score (across experiments):
180009.368666316


Como podemos ver, obtuvimos 5 puntajes, resultado de hacer el experimento 5 veces, para despues hacer un promedio entre esos 5.